# Olist 데이터 이해 (Data Understanding)

## 목적
Tableau Dashboard를 구축하기 전에 Olist 데이터셋의 구조를 이해하고,
각 테이블의 역할과 관계(PK/FK), 주요 컬럼을 파악한다.

또한 각 테이블이 어떤 KPI와 비즈니스 분석에 활용되는지 정리한다.

## 라이브러리 및 데이터베이스 연결

분석에 필요한 라이브러리를 불러오고, Project 3용 SQLite 데이터베이스에 연결한다.

In [10]:
from pathlib import Path
import sqlite3
import pandas as pd

BASE_DIR = Path.cwd().parent
DB_PATH = BASE_DIR / "database" / "olist_dashboard.db"

conn = sqlite3.connect(DB_PATH)

## 데이터베이스 테이블 확인

SQLite 데이터베이스에 저장된 테이블 목록을 확인하여
데이터가 정상적으로 적재되었는지 검증한다.

In [12]:
tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)

print(f"Total Tables: {len(tables)}")

tables

Total Tables: 9


,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category_name_translation
7,products
8,sellers


# Orders 테이블
## 테이블 역할

주문의 전체 생명주기를 관리하는 핵심 테이블이다.

주문 생성부터 승인, 배송, 배송 완료까지의 모든 주문 상태와 시간을 저장한다.

Project 3에서는 주문 수, 주문 상태, 배송 리드타임, 취소율을 계산하는 데 활용한다.
또한 매출 분석 시 주문일 기준을 제공하며, `order_items` 또는 `order_payments` 테이블과 연결하여 월별 매출 추이를 계산한다.

## Primary Key (PK)

- order_id

## Foreign Key (FK)

- customer_id → customers 테이블과 연결

In [15]:
orders = pd.read_sql("""
SELECT *
FROM orders
""", conn)

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [16]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


## 주요 컬럼

- order_id : 주문 ID
- customer_id : 고객 ID
- order_status : 주문 상태
- order_purchase_timestamp : 주문 시각
- order_approved_at : 주문 승인 시각
- order_delivered_carrier_date : 택배사 전달 시각
- order_delivered_customer_date : 고객 배송 완료 시각
- order_estimated_delivery_date : 예상 배송 완료 시각

## 특징

- 주문 생성부터 배송 완료까지의 주문 상태와 시간을 관리하는 핵심 테이블이다.

# Order Items 테이블

## 테이블 역할

주문별 상품 정보를 저장하는 주문 상세 테이블이다.

하나의 주문에 여러 상품이 포함될 수 있기 때문에, 동일한 `order_id`가 여러 행에 나타날 수 있다.

Project 3에서는 상품 판매량, 상품 및 카테고리별 매출, 배송비, 판매자별 성과를 분석하는 데 활용한다.

## Primary Key (PK)

- `order_id` + `order_item_id`  
  두 컬럼의 조합이 각 주문 상품 행을 고유하게 식별하는 복합키 역할을 한다.

## Foreign Key (FK)

- `order_id` → `orders.order_id`
- `product_id` → `products.product_id`
- `seller_id` → `sellers.seller_id`

order_items = pd.read_sql("""
SELECT *
FROM order_items;
""", conn)

order_items.head()

In [22]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


## 주요 컬럼

- `order_id`: 주문 ID
- `order_item_id`: 한 주문 내 상품 순번
- `product_id`: 상품 ID
- `seller_id`: 판매자 ID
- `shipping_limit_date`: 판매자가 상품을 발송해야 하는 기한
- `price`: 상품 가격
- `freight_value`: 배송비

## 특징

- 하나의 주문에는 여러 상품이 포함될 수 있다.
- `order_id`와 `order_item_id`를 함께 사용하여 각 주문 상품을 구분한다.

# Customers 테이블

## 테이블 역할

고객 식별 정보와 고객의 지역 정보를 저장하는 테이블이다.

`customer_id`는 주문별 고객 식별자로 `orders` 테이블과 연결되며,  
`customer_unique_id`는 동일한 실제 고객을 식별하는 데 사용된다.

Project 3에서는 고객 수, 지역별 고객 분포, 재구매 고객 분석에 활용한다.

## Primary Key (PK)

- `customer_id`

`customer_unique_id`는 실제 고객을 식별하는 컬럼이지만, 동일 고객이 여러 주문을 할 수 있으므로 Primary Key로 사용되지 않는다.

재구매 고객 분석이나 고객 수 계산 시에는 `customer_unique_id`를 기준으로 분석한다.

## Foreign Key (FK)

- 별도의 Foreign Key는 없음

In [27]:
customers = pd.read_sql("""
SELECT *
FROM customers;
""", conn)

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [28]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


## 주요 컬럼

- `customer_id`: 주문 단위 고객 ID
- `customer_unique_id`: 실제 고객을 식별하는 고유 ID
- `customer_zip_code_prefix`: 고객 우편번호 앞자리
- `customer_city`: 고객 도시
- `customer_state`: 고객 주

### customer_id와 customer_unique_id의 차이

- customer_id는 주문과 연결되는 ID이다.
- customer_unique_id는 실제 고객을 식별하는 ID이다.
- 재구매 분석은 customer_unique_id 기준으로 수행한다.

## 특징

- `customer_id`는 주문과 연결되는 ID이다.
- `customer_unique_id`는 실제 고객을 식별하는 ID로 재구매 분석에 활용된다.

# Order Payments 테이블

## 테이블 역할

주문별 결제 정보를 저장하는 테이블이다.

하나의 주문은 여러 번에 나누어 결제될 수 있으며,
결제 금액과 결제 방식에 대한 정보를 제공한다.

Project 3에서는 매출, 평균 주문 금액(AOV), 결제 수단 분석 등에 활용한다.

## Primary Key (PK)

- `order_id` + `payment_sequential`

## Foreign Key (FK)

- `order_id` → orders 테이블과 연결

In [33]:
order_payments = pd.read_sql("""
SELECT *
FROM order_payments;
""", conn)

order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [34]:
order_payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


## 주요 컬럼

- `order_id` : 주문 ID
- `payment_sequential` : 동일 주문 내 결제 순번
- `payment_type` : 결제 방식
- `payment_installments` : 할부 개월 수
- `payment_value` : 결제 금액

## 특징

- `customer_id`는 주문과 연결되는 ID이다.
- `customer_unique_id`는 실제 고객을 식별하는 ID로 재구매 분석에 활용된다.

# Order Reviews 테이블

## 테이블 역할

주문에 대한 고객 리뷰 정보를 저장하는 테이블이다.

고객이 남긴 리뷰 점수와 작성 시간을 제공하며,
고객 만족도와 서비스 품질을 분석하는 데 활용된다.

Project 3에서는 평균 리뷰 점수, 리뷰 분포, 고객 만족도 KPI 분석에 활용한다.

## Primary Key (PK)

- `review_id`

## Foreign Key (FK)

- `order_id` → orders 테이블과 연결

In [39]:
order_reviews = pd.read_sql("""
SELECT *
FROM order_reviews;
""", conn)

order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [52]:
order_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


## 주요 컬럼

- `review_id` : 리뷰 ID
- `order_id` : 주문 ID
- `review_score` : 리뷰 평점 (1~5점)
- `review_comment_title` : 리뷰 제목
- `review_comment_message` : 리뷰 내용
- `review_creation_date` : 리뷰 작성 날짜
- `review_answer_timestamp` : 리뷰 등록 시각

## 특징

- 모든 주문이 리뷰를 작성하는 것은 아니다.
- 리뷰 평점(`review_score`)은 모두 존재한다.
- 리뷰 제목과 리뷰 내용은 일부만 작성되어 있다.

# Products 테이블

## 테이블 역할

상품 정보를 저장하는 테이블이다.

상품 카테고리와 상품의 물리적 특성(무게, 크기 등)을 포함하며,
상품 및 카테고리 단위의 분석에 활용된다.

## Primary Key (PK)

- `product_id`

## Foreign Key (FK)

- 없음

In [74]:
products = pd.read_sql("""
SELECT *
FROM products;
""", conn)

products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [76]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


## 주요 컬럼

- `product_id` : 상품 ID
- `product_category_name` : 상품 카테고리
- `product_name_lenght` : 상품명 길이
- `product_description_lenght` : 상품 설명 길이
- `product_photos_qty` : 상품 이미지 개수
- `product_weight_g` : 상품 무게(g)
- `product_length_cm` : 상품 길이(cm)
- `product_height_cm` : 상품 높이(cm)
- `product_width_cm` : 상품 너비(cm)

## 주요 특징

- 상품 카테고리와 상품의 물리적 특성 정보를 제공한다.
- 상품별 분석뿐만 아니라 카테고리별 성과 분석에 활용된다.

# Sellers 테이블

## 테이블 역할

판매자 식별 정보와 판매자의 지역 정보를 저장하는 테이블이다.

`order_items` 테이블과 연결하여 판매자별 매출, 주문 수, 배송 성과를 분석하는 데 활용할 수 있다.

## Primary Key (PK)

- `seller_id`

## Foreign Key (FK)

- 없음

In [83]:
sellers = pd.read_sql("""
SELECT *
FROM sellers;
""", conn)

sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [85]:
sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


## 주요 컬럼

- `seller_id`: 판매자 ID
- `seller_zip_code_prefix`: 판매자 우편번호 앞자리
- `seller_city`: 판매자 도시
- `seller_state`: 판매자 주

# Geolocation 테이블

## 테이블 역할

우편번호를 기준으로 지역의 위도와 경도 정보를 저장하는 테이블이다.

지역별 고객 및 판매자 분포를 지도 시각화하거나 위치 기반 분석에 활용할 수 있다.

## Primary Key (PK)

- 없음

## Foreign Key (FK)

- 없음

In [91]:
geolocation = pd.read_sql("""
SELECT *
FROM geolocation;
""", conn)

geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [93]:
geolocation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB


## 주요 컬럼

- `geolocation_zip_code_prefix`
- `geolocation_lat`
- `geolocation_lng`
- `geolocation_city`
- `geolocation_state`

## 주요 특징

- 동일한 우편번호에 여러 개의 좌표가 존재할 수 있다.

# Product Category Name Translation 테이블

## 테이블 역할

상품 카테고리명을 포르투갈어에서 영어로 변환하기 위한 매핑 테이블이다.

`products.product_category_name`과 연결하여 Tableau 대시보드에서 카테고리명을 영어로 표시하는 데 활용한다.

## Primary Key (PK)

- `product_category_name`

## Foreign Key (FK)

- 없음

In [100]:
product_category_name_translation = pd.read_sql("""
SELECT *
FROM product_category_name_translation;
""", conn)

product_category_name_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [102]:
product_category_name_translation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


## 주요 컬럼

- `product_category_name`: 포르투갈어 상품 카테고리명
- `product_category_name_english`: 영어 상품 카테고리명

## 주요 특징

- 71개의 상품 카테고리명에 대한 번역 정보를 제공한다.
- 실제 매출이나 주문 데이터를 저장하지 않고, 카테고리명을 변환하기 위한 보조 테이블이다.

In [106]:
conn.close()